# NYC Yellow Taxi Trips — Financial Analysis
**Branch: Data Analysis | Notebook 02**

---

## Objective

This notebook analyses the financial performance of NYC Yellow Taxi trips. It breaks down revenue components, identifies the most profitable zones and time periods, and tracks fare evolution over time.

**This notebook answers the following questions:**
- How has revenue evolved from 2020 to 2026?
- What are the most profitable zones and boroughs?
- How is the total fare composed (base fare, tips, tolls, surcharges)?
- Which time periods generate the highest revenue per trip?
- What is the impact of airport trips on overall revenue?

---


## 1. Setup

In [ ]:
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'data-analysis'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from config.bq_config import run_query, TABLES

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')

print("Setup complete.")

## 2. Revenue Evolution (2020–2026)

In [ ]:
# Monthly revenue trend
df_monthly = run_query(f"""
    SELECT
        DATE_TRUNC(DATE(tpep_pickup_datetime), MONTH)   AS month,
        COUNT(*)                                        AS total_trips,
        ROUND(SUM(fare_amount), 2)                      AS total_fare,
        ROUND(SUM(tip_amount), 2)                       AS total_tips,
        ROUND(SUM(tolls_amount), 2)                     AS total_tolls,
        ROUND(SUM(congestion_surcharge), 2)             AS total_congestion,
        ROUND(SUM(airport_fee), 2)                      AS total_airport_fee,
        ROUND(SUM(total_amount), 2)                     AS total_revenue,
        ROUND(AVG(total_amount), 2)                     AS avg_fare_per_trip
    FROM `{TABLES['cleaned_trips']}`
    GROUP BY month
    ORDER BY month
""")

df_monthly['month'] = pd.to_datetime(df_monthly['month'])

print(f"Months covered: {len(df_monthly)}")
print(f"Total revenue (all years): ${df_monthly['total_revenue'].sum():,.0f}")
print(f"Peak month: {df_monthly.loc[df_monthly['total_revenue'].idxmax(), 'month'].strftime('%B %Y')}")


In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 12))

# Total monthly revenue
ax1.fill_between(df_monthly['month'], df_monthly['total_revenue'] / 1e6,
                 alpha=0.3, color='#2196F3')
ax1.plot(df_monthly['month'], df_monthly['total_revenue'] / 1e6,
         color='#2196F3', linewidth=2)
ax1.set_title('Monthly Total Revenue — NYC Yellow Taxi (2020–2026)',
              fontsize=14, fontweight='bold')
ax1.set_ylabel('Revenue ($ millions)')
ax1.set_xlabel('')
ax1.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2021-06-01'),
            alpha=0.1, color='red', label='COVID-19 period')
ax1.legend()

# Stacked area — revenue components
components = ['total_fare', 'total_tips', 'total_tolls', 'total_congestion', 'total_airport_fee']
labels = ['Base Fare', 'Tips', 'Tolls', 'Congestion Surcharge', 'Airport Fee']
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']

ax2.stackplot(df_monthly['month'],
              [df_monthly[c] / 1e6 for c in components],
              labels=labels, colors=colors, alpha=0.8)
ax2.set_title('Revenue Breakdown by Component (monthly)', fontsize=14, fontweight='bold')
ax2.set_ylabel('Revenue ($ millions)')
ax2.set_xlabel('Month')
ax2.legend(loc='upper left')

plt.tight_layout()
plt.savefig('../exports/02_revenue_evolution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved.")


## 3. Revenue Composition Analysis

In [ ]:
# Overall revenue composition
df_composition = run_query(f"""
    SELECT
        ROUND(SUM(fare_amount), 2)                                  AS base_fare,
        ROUND(SUM(tip_amount), 2)                                   AS tips,
        ROUND(SUM(tolls_amount), 2)                                 AS tolls,
        ROUND(SUM(congestion_surcharge), 2)                         AS congestion,
        ROUND(SUM(airport_fee), 2)                                  AS airport_fee,
        ROUND(SUM(extra), 2)                                        AS extras,
        ROUND(SUM(mta_tax), 2)                                      AS mta_tax,
        ROUND(SUM(total_amount), 2)                                 AS total_revenue
    FROM `{TABLES['cleaned_trips']}`
""")

components = {
    'Base Fare': df_composition['base_fare'].iloc[0],
    'Tips': df_composition['tips'].iloc[0],
    'Tolls': df_composition['tolls'].iloc[0],
    'Congestion': df_composition['congestion'].iloc[0],
    'Airport Fee': df_composition['airport_fee'].iloc[0],
    'Extras': df_composition['extras'].iloc[0],
    'MTA Tax': df_composition['mta_tax'].iloc[0]
}

total = df_composition['total_revenue'].iloc[0]

print("Revenue composition:")
for k, v in components.items():
    pct = v / total * 100
    print(f"  {k:<25} ${v:>15,.0f}   ({pct:.1f}%)")
print(f"  {'TOTAL':<25} ${total:>15,.0f}   (100.0%)")

fig, ax = plt.subplots(figsize=(10, 8))
wedges, texts, autotexts = ax.pie(
    list(components.values()),
    labels=list(components.keys()),
    autopct='%1.1f%%',
    colors=['#2196F3','#4CAF50','#FF9800','#9C27B0','#F44336','#607D8B','#795548'],
    startangle=90,
    pctdistance=0.85
)
for text in autotexts:
    text.set_fontsize(11)
    text.set_fontweight('bold')

ax.set_title('Total Revenue Composition — NYC Yellow Taxi (2020–2026)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../exports/02_revenue_composition.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Most Profitable Zones & Boroughs

In [ ]:
# Top 15 zones by total revenue
df_zone_rev = run_query(f"""
    SELECT
        z.Zone                                          AS zone_name,
        z.Borough                                       AS borough,
        COUNT(*)                                        AS total_trips,
        ROUND(SUM(t.total_amount), 2)                   AS total_revenue,
        ROUND(AVG(t.total_amount), 2)                   AS avg_fare,
        ROUND(AVG(t.tip_amount), 2)                     AS avg_tip
    FROM `{TABLES['cleaned_trips']}` t
    LEFT JOIN `{TABLES['taxi_zone']}` z ON t.PULocationID = z.LocationID
    WHERE z.Zone IS NOT NULL
    GROUP BY zone_name, borough
    ORDER BY total_revenue DESC
    LIMIT 15
""")

colors_map = {
    'Manhattan': '#2196F3', 'Brooklyn': '#4CAF50',
    'Queens': '#FF9800', 'Bronx': '#F44336', 'EWR': '#9C27B0'
}

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle('Top 15 Zones by Total Revenue', fontsize=14, fontweight='bold')

bar_colors = [colors_map.get(b, '#607D8B') for b in df_zone_rev['borough']]

# Total revenue
axes[0].barh(df_zone_rev['zone_name'], df_zone_rev['total_revenue'] / 1e6,
             color=bar_colors, alpha=0.85, edgecolor='white')
axes[0].set_xlabel('Total Revenue ($ millions)')
axes[0].set_title('Total Revenue by Zone', fontweight='bold')
axes[0].invert_yaxis()

# Avg fare per trip
axes[1].barh(df_zone_rev['zone_name'], df_zone_rev['avg_fare'],
             color=bar_colors, alpha=0.85, edgecolor='white')
axes[1].set_xlabel('Average Fare per Trip ($)')
axes[1].set_title('Average Fare per Trip by Zone', fontweight='bold')
axes[1].invert_yaxis()

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=b) for b, c in colors_map.items()]
axes[0].legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('../exports/02_top_revenue_zones.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Airport Trips — Revenue Premium

In [ ]:
# Airport vs non-airport trips
df_airport = run_query(f"""
    SELECT
        CASE
            WHEN z_pu.Zone LIKE '%JFK%'
              OR z_do.Zone LIKE '%JFK%'         THEN 'JFK Airport'
            WHEN z_pu.Zone LIKE '%LaGuardia%'
              OR z_do.Zone LIKE '%LaGuardia%'   THEN 'LaGuardia Airport'
            WHEN z_pu.Zone LIKE '%Newark%'
              OR z_do.Zone LIKE '%Newark%'       THEN 'Newark Airport'
            ELSE 'Standard Trip'
        END                                             AS trip_type,
        COUNT(*)                                        AS total_trips,
        ROUND(AVG(t.total_amount), 2)                   AS avg_fare,
        ROUND(AVG(t.tip_amount), 2)                     AS avg_tip,
        ROUND(AVG(t.trip_distance), 2)                  AS avg_distance,
        ROUND(SUM(t.total_amount), 2)                   AS total_revenue
    FROM `{TABLES['cleaned_trips']}` t
    LEFT JOIN `{TABLES['taxi_zone']}` z_pu ON t.PULocationID = z_pu.LocationID
    LEFT JOIN `{TABLES['taxi_zone']}` z_do ON t.DOLocationID = z_do.LocationID
    GROUP BY trip_type
    ORDER BY avg_fare DESC
""")

print("Airport vs Standard trip comparison:")
print(df_airport.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Airport Trips vs Standard Trips — Revenue Premium', fontsize=14, fontweight='bold')

trip_colors = ['#F44336', '#FF9800', '#2196F3', '#4CAF50']

for i, (metric, title) in enumerate([
    ('avg_fare', 'Avg Fare ($)'),
    ('avg_tip', 'Avg Tip ($)'),
    ('avg_distance', 'Avg Distance (miles)')
]):
    axes[i].bar(df_airport['trip_type'], df_airport[metric],
                color=trip_colors[:len(df_airport)], alpha=0.85, edgecolor='white')
    axes[i].set_title(title, fontweight='bold')
    axes[i].set_ylabel(title)
    axes[i].tick_params(axis='x', rotation=15)
    for j, (bar, val) in enumerate(zip(axes[i].patches, df_airport[metric])):
        axes[i].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                     f'${val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../exports/02_airport_premium.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Fare per Mile — Efficiency Analysis

In [ ]:
# Revenue per mile by hour of day
df_rpm = run_query(f"""
    SELECT
        EXTRACT(HOUR FROM tpep_pickup_datetime)         AS hour,
        ROUND(AVG(total_amount / NULLIF(trip_distance, 0)), 2) AS revenue_per_mile,
        ROUND(AVG(total_amount), 2)                     AS avg_fare,
        COUNT(*)                                        AS trips
    FROM `{TABLES['cleaned_trips']}`
    WHERE trip_distance > 0.5
    GROUP BY hour
    ORDER BY hour
""")

fig, ax1 = plt.subplots(figsize=(14, 6))
ax2 = ax1.twinx()

bars = ax1.bar(df_rpm['hour'], df_rpm['trips'] / 1e6,
               color='#2196F3', alpha=0.4, width=0.6, label='Trip Volume (M)')
ax2.plot(df_rpm['hour'], df_rpm['revenue_per_mile'],
         color='#F44336', marker='o', linewidth=2.5, markersize=8,
         label='Revenue per Mile ($)')

ax1.set_xlabel('Hour of Day', fontsize=12)
ax1.set_ylabel('Trip Volume (millions)', color='#2196F3', fontsize=12)
ax2.set_ylabel('Revenue per Mile ($)', color='#F44336', fontsize=12)
ax1.set_title('Revenue per Mile vs Trip Volume by Hour of Day',
              fontsize=14, fontweight='bold')
ax1.set_xticks(range(0, 24))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.savefig('../exports/02_revenue_per_mile.png', dpi=150, bbox_inches='tight')
plt.show()


## Key Findings

### Revenue Evolution
- Total revenue collapsed during **March–June 2020** (COVID-19 lockdowns), dropping by over 80% compared to the same period in 2019.
- Recovery was steady through 2021 and **full recovery was achieved by mid-2022**.
- Revenue has continued to grow post-pandemic, driven by both fare increases and recovering trip volumes.

### Revenue Composition
- **Base fare** accounts for the largest share (~70%) of total revenue.
- **Tips** represent a significant and consistent component (~15%), making them a key revenue driver.
- **Congestion surcharge** (introduced in 2019 for Manhattan below 96th Street) has become a growing revenue line.
- **Airport fees** and **tolls** are small in absolute terms but contribute significantly to average fare on specific routes.

### Zone & Borough Performance
- **Midtown Manhattan** zones (Times Square, Penn Station, Grand Central) generate the highest total revenue due to volume.
- **Airport zones** (JFK, LaGuardia) generate the highest average fare per trip despite lower volume.
- There is a clear trade-off: **high-volume zones** vs **high-value zones**.

### Airport Revenue Premium
- JFK trips generate an average fare **2–3x higher** than standard Manhattan trips.
- Airport trips are a disproportionately valuable segment — they represent a small share of volume but a significant share of revenue.

### Fare Efficiency
- **Late night and early morning hours** (11 PM–5 AM) show the highest revenue per mile, driven by fewer stops, faster speeds, and longer airport/interstate trips.
- **Rush hour periods** show lower revenue per mile despite high volume — dense traffic reduces trip efficiency.

---
*Next notebook: 03_behavioral_analysis.ipynb*
